In [1]:
# import
import tensorflow as tf


# パス・画像サイズ・バッチサイズ
train_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/train",
    image_size = (224,224),
    label_mode = "binary",
    batch_size = 150,
    shuffle = True
)

test_dataset = tf.keras.preprocessing.image_dataset_from_directory(
    "dog_cat_photos/test",
    image_size = (224,224),
    label_mode = "binary",
    batch_size = 50,
    shuffle = False
)


# class_names確認
print(train_dataset.class_names)


# 水増し作成
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
])


# MobileNetV2作成・凍結
base_model = tf.keras.applications.mobilenet_v2.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet",
    pooling="avg"
)
base_model.trainable = False


# 分類モデル作成
input_layer = tf.keras.Input(shape=(224, 224, 3))

x = data_augmentation(input_layer)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x)
x = tf.keras.layers.Dropout(0.2)(x)

output_layer = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs=input_layer, outputs=output_layer)


# compile
model.compile(optimizer="adam",
              loss='binary_crossentropy',
              metrics=["accuracy"])


# fit
model.fit(train_dataset, epochs=5)


# evaluate
model.evaluate(test_dataset)


Found 300 files belonging to 2 classes.
Found 100 files belonging to 2 classes.
['cat', 'dog']
Epoch 1/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 22s 8s/step - accuracy: 0.5867 - loss: 0.6896
Epoch 2/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 12s 6s/step - accuracy: 0.7033 - loss: 0.5881
Epoch 3/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 12s 6s/step - accuracy: 0.7800 - loss: 0.5023
Epoch 4/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 12s 6s/step - accuracy: 0.8133 - loss: 0.4357
Epoch 5/5
2/2 ━━━━━━━━━━━━━━━━━━━━ 11s 6s/step - accuracy: 0.8767 - loss: 0.3636
2/2 ━━━━━━━━━━━━━━━━━━━━ 7s 2s/step - accuracy: 0.9400 - loss: 0.2781


[0.2780764102935791, 0.9399999976158142]